In [ ]:
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import data from huggingface

In [ ]:
from datasets import load_dataset

tawnza_data = load_dataset("Tamazight-NLP/TAWNZA_TIFINAGH_ARABIC_DATASET")

print(tawnza_data)
print(tawnza_data["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/319 [00:00<?, ?B/s]

DATASET TAWNZA SEASON 1.json:   0%|          | 0.00/37.5M [00:00<?, ?B/s]

DATASET%20TAWNZA%20SEASON%202.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/131624 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'season', 'episode', 'line', 'source', 'target', 'start', 'end', 'duration_ms'],
        num_rows: 131624
    })
})
{'id': 'S01_EP02_LINE1', 'season': 1, 'episode': 2, 'line': 1, 'source': 'مرحبا بكم مشاهدي قناة تمازيغت أينما كنتم', 'target': 'ⴱⵔⵔⴽⴰⵜ ⵉⵎⴰⵏⵏⴰⵢⵏ ⵏ ⵜⴱⴰⴷⵓⵜ ⵜⴰⵎⴰⵣⵉⵖⵜ ⵖⵉⵏⵏⴰ ⴳ ⵜⵍⵍⴰⵎ', 'start': '00:00:39,080', 'end': '00:00:43,160', 'duration_ms': 4079}


In [ ]:
import json
from datasets import Dataset

formatted_data = []
for item in tawnza_data["train"]:
    formatted_data.append({
        "arabe": item["source"],
        "tamazight": item["target"]
    })


# 3. Créer le Dataset Hugging Face et séparer train/validation (90/10)
full_dataset = Dataset.from_list(formatted_data)
dataset_split = full_dataset.train_test_split(test_size=0.1, seed=42)
print(dataset_split)

DatasetDict({
    train: Dataset({
        features: ['arabe', 'tamazight'],
        num_rows: 118461
    })
    test: Dataset({
        features: ['arabe', 'tamazight'],
        num_rows: 13163
    })
})


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

model_checkpoint = "facebook/nllb-200-distilled-600M"

# Initialisation avec les codes de langue stricts
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, src_lang="arb_Arab", tgt_lang="zgh_Tfng")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# Nouvelle fonction de prétraitement (Mise à jour Transformers v4.30+)
def preprocess_function(examples):
    inputs = examples["arabe"]
    targets = examples["tamazight"]

    model_inputs = tokenizer(
        text=inputs,
        text_target=targets,
        max_length=128,
        truncation=True
    )

    return model_inputs

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Appliquer le prétraitement
tokenized_datasets = dataset_split.map(preprocess_function, batched=True)

# Définition des hyperparamètres d'entraînement
# Définition optimisée pour VRAM limitée (GPU T4)
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(15000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

print(f"Taille du nouveau dataset : {len(small_train_dataset)} phrases")

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-ar-zgh-model",
    eval_strategy="steps",             # Évaluer régulièrement
    eval_steps=200,                    # Toutes les 200 étapes
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="adafactor",
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,                # <-- Réduit à 1 seule époque
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False,
)

# 3. Lancement avec les petits datasets
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset, # <-- Utilisation du subset
    eval_dataset=small_eval_dataset,   # <-- Utilisation du subset
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

# Sauvegarde
model.save_pretrained("./api_model")
tokenizer.save_pretrained("./api_model")
print("Modèle exporté avec succès.")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

...

Step,Training Loss,Validation Loss
200,No log,1.887441
400,No log,1.698285
600,17.077164,1.608530
800,17.077164,1.567226


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle exporté avec succès.
